In [111]:
import pandas as pd
import psycopg

In [112]:
predictions = pd.read_csv("../data/nq_first_to_100_predictions_cumulative.csv")
# predictions = pd.read_csv("../data/weekly_trade_log_2026_week_5_with_auction_direction.csv")
predictions = predictions.dropna(how="all").reset_index(drop=True)

predictions["Date"] = pd.to_datetime(predictions["Date"])

study_dates = predictions["Date"].dt.date.tolist()

In [113]:
original_evaluated = pd.read_csv(
    "../data/nq_first_to_100_predictions_evaluated.csv"
)

original_evaluated["Date"] = pd.to_datetime(original_evaluated["Date"])

In [114]:
conn = psycopg.connect("dbname=dailyedge_development")

In [115]:
test_date = "2026-04-27"

rth = pd.read_sql(
    """
    SELECT timestamp, open, high, low, close, volume
    FROM CANDLES
    WHERE timestamp::date = %s
      AND timestamp::time BETWEEN '08:30:00' AND '15:15:00'
    ORDER BY timestamp
    """,
    conn,
    params=(test_date,)
)

/tmp/ipykernel_46890/2509636352.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rth = pd.read_sql(


In [116]:
rth.iloc[0]

timestamp    2026-04-27 08:30:00
open                27684.152457
high                27708.644333
low                 27635.673692
close               27652.085774
volume                      6458
Name: 0, dtype: object

In [117]:
opening_price = rth.iloc[0]["open"]

upper_target = opening_price + 100
lower_target = opening_price - 100

opening_price, upper_target, lower_target

(np.float64(27684.152457), np.float64(27784.152457), np.float64(27584.152457))

In [118]:
target_hits = rth[
    (rth["high"] >= upper_target) |
    (rth["low"] <= lower_target)
]

first_hit = target_hits.iloc[0]

first_hit

timestamp    2026-04-27 08:51:00
open                27597.042176
high                27602.092047
low                 27574.065261
close               27577.852664
volume                      2783
Name: 21, dtype: object

In [119]:
hit_upper = first_hit["high"] >= upper_target
hit_lower = first_hit["low"] <= lower_target

hit_upper, hit_lower

(np.False_, np.True_)

In [120]:
if hit_upper and hit_lower:
    result = "Ambiguous"
elif hit_upper:
    result = "Long"
elif hit_lower:
    result = "Short"

result

'Short'

In [121]:
test_date = "2026-04-27"

holiday_rth = pd.read_sql(
    """
    SELECT timestamp, open, high, low, close, volume
    FROM CANDLES
    WHERE timestamp::date = %s
      AND timestamp::time BETWEEN '08:30:00' AND '15:15:00'
    ORDER BY timestamp
    """,
    conn,
    params=(test_date,)
)

holiday_rth

/tmp/ipykernel_46890/951943134.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  holiday_rth = pd.read_sql(


,timestamp,open,high,low,close,volume
0,2026-04-27 08:30:00,27684.152457,27708.644333,27635.673692,27652.085774,6458
1,2026-04-27 08:31:00,27653.095748,27662.690504,27635.168705,27659.408087,2681
2,2026-04-27 08:32:00,27659.913074,27669.507830,27645.773435,27662.438010,2044
3,2026-04-27 08:33:00,27662.690504,27666.730401,27639.208602,27650.823306,1744
4,2026-04-27 08:34:00,27651.075800,27655.620684,27637.441147,27641.986031,1467
...,...,...,...,...,...,...
401,2026-04-27 15:11:00,27701.322020,27705.361917,27701.069526,27702.836981,191
402,2026-04-27 15:12:00,27702.584488,27709.149321,27702.584488,27708.896827,254
403,2026-04-27 15:13:00,27708.644333,27708.896827,27698.544591,27699.554565,227
404,2026-04-27 15:14:00,27699.807058,27700.817033,27697.282123,27700.817033,252


In [122]:
(holiday_rth["timestamp"].dt.time == pd.Timestamp("15:15").time()).any()

np.True_

In [133]:
def evaluate_session(target_date):
    rth = pd.read_sql(
        """
        SELECT timestamp, open, high, low, close, volume
        FROM CANDLES
        WHERE timestamp::date = %s
          AND timestamp::time BETWEEN '08:30:00' AND '15:15:00'
        ORDER BY timestamp
        """,
        conn,
        params=(target_date,)
    )

    rth["timestamp"] = pd.to_datetime(rth["timestamp"])

    # Validate that this is a full RTH session
    has_open = (
        rth["timestamp"].dt.time == pd.Timestamp("08:30").time()
    ).any()

    has_close = (
        rth["timestamp"].dt.time == pd.Timestamp("15:15").time()
    ).any()

    if not has_open or not has_close:
        return "Invalid", None

    # 08:30 reference price and ±100 targets
    opening_price = rth.loc[
        rth["timestamp"].dt.time == pd.Timestamp("08:30").time(),
        "open"
    ].iloc[0]

    upper_target = opening_price + 70
    lower_target = opening_price - 70

    # Find candles that reach either ±100 target
    target_hits = rth[
        (rth["high"] >= upper_target) |
        (rth["low"] <= lower_target)
    ]

    if target_hits.empty:
        return "Neither", None

    first_hit = target_hits.iloc[0]

    hit_upper = first_hit["high"] >= upper_target
    hit_lower = first_hit["low"] <= lower_target

    # Both ±100 targets inside the same 1-minute candle
    if hit_upper and hit_lower:
        return "Ambiguous", "Unknown"

    result = "Long" if hit_upper else "Short"

    # Determine whether opposite 50 was reached before target candle
    before_target = rth[
        rth["timestamp"] < first_hit["timestamp"]
    ]

    if result == "Long":
        opposite_50_before = (
            before_target["low"].min() <= opening_price - 35
        )
        opposite_50_in_target_candle = (
            first_hit["low"] <= opening_price - 35
        )
    else:
        opposite_50_before = (
            before_target["high"].max() >= opening_price + 35
        )
        opposite_50_in_target_candle = (
            first_hit["high"] >= opening_price + 35
        )

    # Classify cleanliness of the ±100 move
    if opposite_50_before:
        clean_move = "Not Clean"
    elif opposite_50_in_target_candle:
        clean_move = "Unknown"
    else:
        clean_move = "Clean"

    return result, clean_move

In [134]:
for i, date in enumerate(study_dates):
    if pd.isna(date):
        print("Index:", i)

In [135]:
results = []

for date in study_dates:
    result, clean_move = evaluate_session(date.strftime("%Y-%m-%d"))

    results.append({
        "Date": date,
        "Result": result,
        "Clean Move": clean_move
    })

outcomes = pd.DataFrame(results)

/tmp/ipykernel_46890/3776166059.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rth = pd.read_sql(
/tmp/ipykernel_46890/3776166059.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rth = pd.read_sql(
/tmp/ipykernel_46890/3776166059.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rth = pd.read_sql(
/tmp/ipykernel_46890/3776166059.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider

In [136]:
predictions["Result"] = outcomes["Result"].values

predictions["Correct"] = (
    predictions["Bias"] == predictions["Result"]
).where(
    predictions["Result"].isin(["Long", "Short"]),
    pd.NA
).astype("boolean")

In [137]:
predictions["Correct"].value_counts(dropna=False)

Correct
True     118
False     90
<NA>      11
Name: count, dtype: Int64

In [138]:
comparison = predictions[["Date", "Correct"]].merge(
    original_evaluated[["Date", "Correct"]],
    on="Date",
    suffixes=("_new", "_original")
)

comparison[
    comparison["Correct_new"] != comparison["Correct_original"]
]

,Date,Correct_new,Correct_original
2,2026-04-29,True,False
5,2026-05-04,True,False
21,2026-05-26,False,True
28,2026-06-04,False,True
36,2026-06-16,False,True
42,2026-06-24,True,False
45,2026-06-29,False,True
51,2026-07-07,False,True


In [139]:
clean_breakdown = outcomes["Clean Move"].value_counts(dropna=True)

clean_percentages = (
    outcomes["Clean Move"]
    .value_counts(normalize=True, dropna=True)
    .mul(100)
    .round(2)
)

pd.DataFrame({
    "Total": clean_breakdown,
    "Percentage": clean_percentages
})

,Total,Percentage
Clean Move,,
Clean,141,67.79
Not Clean,67,32.21


In [130]:
output_path = "../data/nq_first_to_100_predictions_evaluated.csv"

predictions.to_csv(output_path, index=False)

In [131]:
# display(
#     predictions[
#         ["Date", "Day", "Bias", "Confidence", "Auction Direction", "Context", "Result", "Correct"]
#     ]
# )

accuracy = predictions["Correct"].mean() * 100

print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 57.56%


In [132]:
valid_predictions = predictions.dropna(subset=["Correct"])

accuracy = valid_predictions["Correct"].mean() * 100

print(f"Overall accuracy: {accuracy:.2f}%")
print(f"Correct: {valid_predictions['Correct'].sum()} / {len(valid_predictions)}")

Overall accuracy: 57.56%
Correct: 118 / 205
